# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will walk through steps to load the data, get an overview of the record sets and fields (by their `@id`), extract tabular data, and apply some exploratory analyses.

### Dataset Source
The FAIR² dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print('Dataset name:', metadata.name)
print('Description:', metadata.description)
print('License:', getattr(metadata, 'license', None))

## 2. Data Overview
List all available record sets and their IDs in the dataset. For each record set, display the fields and their `@id`s and descriptions (where available).

In [ ]:
# List record sets and their fields using their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f'Record Set @id: {rs.id}')
        print(f'  Name: {getattr(rs, "name", "(no name)")}'.strip())
        print(f'  Description: {getattr(rs, "description", "(no description)")}'.strip())
        print("  Fields:")
        for field in rs.fields:
            print(f'    - @id: {field.id}')
            name = getattr(field, 'name', None)
            if name:
                print(f'        name: {name}')
            desc = getattr(field, 'description', None)
            if desc:
                print(f'        description: {desc}')
        print()

## 3. Data Extraction
Load each available record set into a pandas DataFrame. If the dataset contains multiple record sets, this step will create a DataFrame for each, referenced by their `@id`s. We'll print the column (field) names (as their `@id`s) and the first few records.

In [ ]:
# Build a dictionary of DataFrames for each record set
dataframes = dict()
import warnings; warnings.filterwarnings('ignore', category=UserWarning) # suppress possible dtype warnings
if not record_sets:
    print("No record sets available in this dataset for extraction.")
else:
    for rs in record_sets:
        recs = list(dataset.records(record_set=rs.id))
        dataframes[rs.id] = pd.DataFrame(recs)
        print(f"Loaded DataFrame for record set @id={rs.id} with columns:")
        print(list(dataframes[rs.id].columns))
        print(dataframes[rs.id].head(), "\n")
    # For demonstration, pick the first record set for subsequent analysis
    main_record_set_id = record_sets[0].id

## 4. Exploratory Data Analysis (EDA)
Apply example processing: filter numeric fields, normalize columns, and groupby operations. Use field `@id` as column names. Adjust this cell for your own analysis based on the field `@id`s printed above.

In [ ]:
# Example: Find a numeric field and perform filtering, normalization, and grouping

df = dataframes[main_record_set_id]
numeric_field_id = None
group_field_id = None
# Try to pick numeric and group fields automatically (simple heuristic)
for c in df.columns:
    # Try if the column can be converted to numeric
    try:
        if pd.api.types.is_numeric_dtype(df[c]) or pd.to_numeric(df[c].dropna(), errors='coerce').notnull().any():
            numeric_field_id = c
            break
    except Exception:
        continue
for c in df.columns:
    if c != numeric_field_id and df[c].nunique() < 8:
        group_field_id = c
        break
if not numeric_field_id:
    print("No obvious numeric field found for filtering.")
else:
    print(f"Using numeric field @id: {numeric_field_id}")
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a group field (categorical)
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and if available, its distribution across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_field_id or df[numeric_field_id].isnull().all():
    print("No numeric field available for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, overview, and perform basic exploratory analysis on a FAIR²-structured dataset using the `mlcroissant` library. Each entity, field, and record is referenced by its `@id` as required by the Croissant standard.

You can build upon this template to apply further analytics, feature engineering, and modeling to Croissant-packaged datasets.